# COMP3710 Lab 2 - Part 1

The original teacher-provided code is kept unchanged below. The modified PyTorch/GPU solution is placed in separate code cells afterwards.

**Google Colab:** select `Runtime > Change runtime type > T4 GPU` before running the modified solution cells.


## Original teacher-provided code

Run these cells first to demonstrate the original Part 1 task.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Set parameters for the signal
N = 2048               # Number of sample points
T = 1.0                # Duration of the signal in seconds
f0 = 1                 # Fundamental frequency of the square wave in Hz

# List of harmonic numbers used to construct the square wave
harmonics = [1, 3, 5]

# Define the square wave function
def square_wave(t):
    return np.sign(np.sin(2.0 * np.pi * f0 * t))

# Fourier series approximation of the square wave
def square_wave_fourier(t, f0, N):
    result = np.zeros_like(t)
    for k in range(N):
        # The Fourier series of a square wave contains only odd harmonics.
        n = 2 * k + 1
        # Add harmonics to reconstruct the square wave.
        result += np.sin(2 * np.pi * n * f0 * t) / n
    return (4 / np.pi) * result

# Create the time vector
# np.linspace generates evenly spaced numbers over a specified interval.
# We use endpoint=False because the interval is periodic.
t = np.linspace(0.0, T, N, endpoint=False)

# Generate the original square wave
square = square_wave(t)

plt.figure(figsize=(12, 8))
# Plot the original square wave
plt.subplot(2, 3, 1)
plt.plot(t, square, 'k', label="Square wave")
plt.title("Original Square Wave")
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.legend()

# Plot Fourier reconstructions under different number of harmonics
for i, Nh in enumerate(harmonics, start=2):
    plt.subplot(2, 3, i)
    y = square_wave_fourier(t, f0, Nh)
    plt.plot(t, y, label=f"N={Nh} harmonics")
    plt.plot(t, square, 'k--', alpha=0.5, label="Square wave")
    plt.title(f"Fourier Approximation with N={Nh}")
    plt.ylim(-1.5, 1.5)
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 2. Apply the DFT and time the execution

def naive_dft(x):
    """
    Compute the Discrete Fourier Transform (DFT) of a 1D signal.

    This is a "naive" implementation that directly follows the DFT formula,
    which has a time complexity of O(N^2).

    Args:
        x (np.ndarray): The input signal, a 1D NumPy array.

    Returns:
        np.ndarray: The complex-valued DFT of the input signal.
    """
    N = len(x)
    # Create an empty array of complex numbers to store the DFT results
    X = np.zeros(N, dtype=np.complex128)

    # Iterate through each frequency bin (k)
    for k in range(N):
        # For each frequency bin, sum the contributions from all input samples (n)
        for n in range(N):
            # The core DFT formula: x[n] * e^(-2j * pi * k * n / N)
            angle = -2j * np.pi * k * n / N
            X[k] += x[n] * np.exp(angle)

    return X

# Construct a square wave using 50 harmonics
signal = square_wave_fourier(t, f0, 50)
# Time the naive DFT implementation
start_time_naive = time.time()
dft_result = naive_dft(signal)
end_time_naive = time.time()
naive_duration = end_time_naive - start_time_naive

# Time NumPy's FFT implementation
start_time_fft = time.time()
fft_result = np.fft.fft(signal)
end_time_fft = time.time()
fft_duration = end_time_fft - start_time_fft

# 3. Print Timings and Verification
print("--- DFT/FFT Performance Comparison ---")
print(f"Naive DFT Execution Time: {naive_duration:.6f} seconds")
print(f"NumPy FFT Execution Time: {fft_duration:.6f} seconds")
# It is possible for the FFT to be so fast that the duration is 0.0.
if fft_duration > 0:
    print(f"FFT is approximately {naive_duration / fft_duration:.2f} times faster.")
else:
    print("FFT was too fast to measure a significant duration difference.")

# Check if our implementation is close to NumPy's result
print(f"\nOur DFT implementation is close to NumPy's FFT: {np.allclose(dft_result, fft_result)}")

# 4. Prepare for Plotting
xf = np.fft.fftfreq(N, d=T/N)[:N//2]
magnitude = 2.0/N * np.abs(dft_result[0:N//2])

# 5. Visualize the Results
plt.style.use('seaborn-v0_8-darkgrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot the original time-domain signal
ax1.plot(t, signal, color='c')
ax1.set_title('Input Sine Wave Signal', fontsize=16)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Amplitude', fontsize=12)
ax1.set_xlim(0, 1.0)
ax1.grid(True)

# Plot the frequency-domain signal (magnitude of the DFT)
ax2.stem(xf, magnitude, basefmt=" ")
ax2.set_title('Discrete Fourier Transform (Magnitude Spectrum)', fontsize=16)
ax2.set_xlabel('Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Magnitude', fontsize=12)
ax2.set_xlim(0, 50)
ax2.grid(True)

# Add vertical lines for the first ten frequencies
for i in range(20):
    if i < len(xf) and i % 2 == 1:
        ax2.axvline(
            xf[i], color='r', linestyle='--', alpha=0.7,
            label=f'f{i}: {i}* f0 = {xf[i]:.1f} Hz'
        )

ax2.legend()
plt.tight_layout()
plt.show()


## Modified PyTorch and GPU solution

The following cells are the separate modified implementation. The original functions are not overwritten above.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import torch

# Set parameters for the signal
N = 2048               # Number of sample points
T = 1.0                # Duration of the signal in seconds
f0 = 1                 # Fundamental frequency of the square wave in Hz

# List of harmonic numbers used to construct the square wave
# Add 20 and 50 harmonics to observe the effect of higher-order harmonics
harmonics = [1, 3, 5, 20, 50]

# Original NumPy square_wave implementation (preserved and commented out)
# def square_wave(t):
#     return np.sign(np.sin(2.0 * np.pi * f0 * t))

# PyTorch version using only torch tensor operations
def square_wave(t):
    return torch.sign(torch.sin(2.0 * torch.pi * f0 * t))

# Original NumPy square_wave_fourier implementation (preserved and commented out)
# def square_wave_fourier(t, f0, N):
#     result = np.zeros_like(t)
#     for k in range(N):
#         n = 2 * k + 1
#         result += np.sin(2 * np.pi * n * f0 * t) / n
#     return (4 / np.pi) * result

# PyTorch version using broadcasting to calculate all odd harmonics at once
def square_wave_fourier(t, f0, num_harmonics):
    # Generate the odd harmonic numbers 1, 3, 5, ...
    odd_harmonics = 2 * torch.arange(num_harmonics, device=t.device) + 1
    # Reshape the harmonic and time vectors for broadcasting
    n = odd_harmonics[:, None].to(t.dtype)
    angles = 2 * torch.pi * n * f0 * t[None, :]
    return (4 / torch.pi) * torch.sum(torch.sin(angles) / n, dim=0)

# Create the time vector
# np.linspace generates evenly spaced numbers over a specified interval.
# We use endpoint=False because the interval is periodic.
# torch.arange excludes the period endpoint, equivalent to endpoint=False
t = torch.arange(N, dtype=torch.float64) * (T / N)

# Generate the original square wave
square = square_wave(t)
# Convert to NumPy only for plotting; the function calculations remain in PyTorch
t_plot = t.cpu().numpy()
square_plot = square.cpu().numpy()

plt.figure(figsize=(12, 8))
# Plot the original square wave
plt.subplot(2, 3, 1)
plt.plot(t_plot, square_plot, 'k', label="Square wave")
plt.title("Original Square Wave")
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.legend()
# Plot Fourier reconstructions under different number of harmonics
for i, Nh in enumerate(harmonics, start=2):
    plt.subplot(2, 3, i)
    y = square_wave_fourier(t, f0, Nh)
    plt.plot(t_plot, y.cpu().numpy(), label=f"N={Nh} harmonics")
    plt.plot(t_plot, square_plot, 'k--', alpha=0.5, label="Square wave")
    plt.title(f"Fourier Approximation with N={Nh}")
    plt.ylim(-1.5, 1.5)
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2. Apply the DFT and time the execution

# Original NumPy naive_dft implementation (preserved and commented out)
# def naive_dft(x):
#     N = len(x)
#     X = np.zeros(N, dtype=np.complex128)
#     for k in range(N):
#         for n in range(N):
#             angle = -2j * np.pi * k * n / N
#             X[k] += x[n] * np.exp(angle)
#     return X

def synchronize_device(device):
    """Synchronize asynchronous GPU work for accurate timing."""
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    elif device.type == 'mps':
        torch.mps.synchronize()

def naive_dft_torch(x, device):
    """Explicit O(N^2) DFT using PyTorch tensors without torch.fft."""
    # Use double precision on CPU for verification and float32 on GPU for compatibility
    real_dtype = torch.float64 if device.type == 'cpu' else torch.float32
    x = x.to(device=device, dtype=real_dtype)
    num_samples = x.numel()
    # k represents frequency bins and n represents input sample positions
    k = torch.arange(num_samples, device=device, dtype=real_dtype)[:, None]
    n = torch.arange(num_samples, device=device, dtype=real_dtype)[None, :]
    # Explicitly calculate the cosine real part and sine imaginary part without an FFT
    angles = -2 * torch.pi * k * n / num_samples
    real_part = torch.cos(angles) @ x
    imag_part = torch.sin(angles) @ x
    return torch.complex(real_part, imag_part)

def get_gpu_device():
    """Select CUDA first, then Apple MPS, or return None if no GPU is available."""
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return None

# Construct a square wave using 50 harmonics
signal = square_wave_fourier(t, f0, 50).to(torch.float32)
signal_numpy = signal.cpu().numpy()

# Warm up both methods so one-time library initialization is excluded from timing
_ = naive_dft_torch(signal[:64], torch.device('cpu'))
_ = np.fft.fft(signal_numpy[:64])

# Time the explicit PyTorch DFT on the CPU
cpu_device = torch.device('cpu')
start_time_naive = time.time()
dft_result_torch = naive_dft_torch(signal, cpu_device)
end_time_naive = time.time()
naive_duration = end_time_naive - start_time_naive
dft_result = dft_result_torch.cpu().numpy()

# Time NumPy's FFT implementation
start_time_fft = time.time()
fft_result = np.fft.fft(signal_numpy)
end_time_fft = time.time()
fft_duration = end_time_fft - start_time_fft

# Time the explicit PyTorch DFT on an available GPU
gpu_device = get_gpu_device()
gpu_duration = None
gpu_dft_result = None
if gpu_device is not None:
    synchronize_device(gpu_device)
    start_time_gpu = time.time()
    gpu_dft_result = naive_dft_torch(signal, gpu_device)
    synchronize_device(gpu_device)
    gpu_duration = time.time() - start_time_gpu

# 3. Print Timings and Verification
print("--- DFT/FFT Performance Comparison ---")
print(f"PyTorch CPU Naive DFT Execution Time: {naive_duration:.6f} seconds")
print(f"NumPy FFT Execution Time: {fft_duration:.6f} seconds")
# It's possible for the FFT to be so fast that the duration is 0.0, so we handle that case.
if fft_duration > 0:
    print(f"FFT is approximately {naive_duration / fft_duration:.2f} times faster.")
else:
    print("FFT was too fast to measure a significant duration difference.")
if gpu_duration is not None:
    print(f"PyTorch GPU Naive DFT ({gpu_device}) Execution Time: {gpu_duration:.6f} seconds")
else:
    print("PyTorch GPU Naive DFT: skipped because no CUDA/MPS GPU is available.")

# Check if our implementation is close to NumPy's result
# np.allclose is used for comparing floating-point arrays.
# The CPU result uses complex128, so NumPy's default tolerance is sufficient
cpu_matches_fft = np.allclose(dft_result, fft_result)
print(f"\nPyTorch CPU DFT is close to NumPy's FFT: {cpu_matches_fft}")
if gpu_dft_result is not None:
    gpu_matches_fft = np.allclose(
        gpu_dft_result.cpu().numpy(), fft_result, rtol=1e-4, atol=1e-4
    )
    print(f"PyTorch GPU DFT is close to NumPy's FFT: {gpu_matches_fft}")

# 4. Prepare for Plotting
# Generate the frequency axis for the plot.
# np.fft.fftfreq returns the DFT sample frequencies.
# We only need the first half of the frequencies (the positive ones) due to symmetry.
xf = np.fft.fftfreq(N, d=T/N)[:N//2]
# We normalize the magnitude by N and multiply by 2 to get the correct amplitude.
magnitude = 2.0/N * np.abs(dft_result[0:N//2])

# 5. Visualize the Results
plt.style.use('seaborn-v0_8-darkgrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot the original time-domain signal
ax1.plot(t_plot, signal_numpy, color='c')
ax1.set_title('Input Square Wave Signal', fontsize=16)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Amplitude', fontsize=12)
ax1.set_xlim(0, 1.0) # Show a few cycles of the sine wave
ax1.grid(True)

# Plot the frequency-domain signal (magnitude of the DFT)
ax2.stem(xf, magnitude, basefmt=" " )
ax2.set_title(
    'Discrete Fourier Transform (Magnitude Spectrum)',
    fontsize=16
)
ax2.set_xlabel('Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Magnitude', fontsize=12)
ax2.set_xlim(0, 50) # Focus on lower frequencies
ax2.grid(True)

# Add vertical lines for the first ten frequencies
for i in range(20):
    if i < len(xf) and i % 2 == 1:  # Only plot odd harmonics
        ax2.axvline(
            xf[i], color='r', linestyle='--', alpha=0.7,
            label=f'f{i}: {i}* f0 = {xf[i]:.1f} Hz'
        )

# Only show labels for first 3 frequencies to avoid cluttering
ax2.legend()

plt.tight_layout()
plt.show()

# 6. Change the data size and compare the execution times of all three methods
# Start with smaller sizes because the explicit O(N^2) DFT uses substantial memory
data_sizes = [256, 512, 1024, 2048]
timing_results = []

for data_size in data_sizes:
    t_size = torch.arange(data_size, dtype=torch.float64) * (T / data_size)
    signal_size = square_wave_fourier(t_size, f0, 50).to(torch.float32)
    signal_size_numpy = signal_size.cpu().numpy()

    start = time.perf_counter()
    _ = naive_dft_torch(signal_size, cpu_device)
    cpu_time = time.perf_counter() - start

    start = time.perf_counter()
    _ = np.fft.fft(signal_size_numpy)
    fft_time = time.perf_counter() - start

    gpu_time = None
    if gpu_device is not None:
        synchronize_device(gpu_device)
        start = time.perf_counter()
        _ = naive_dft_torch(signal_size, gpu_device)
        synchronize_device(gpu_device)
        gpu_time = time.perf_counter() - start

    timing_results.append((data_size, fft_time, cpu_time, gpu_time))

print("\n--- Timing comparison for different data sizes ---")
print(f"{'N':>6} {'NumPy FFT (s)':>16} {'Torch CPU DFT (s)':>20} {'Torch GPU DFT (s)':>20}")
for data_size, fft_time, cpu_time, gpu_time in timing_results:
    gpu_text = f"{gpu_time:.6f}" if gpu_time is not None else 'N/A'
    print(f"{data_size:>6} {fft_time:>16.6f} {cpu_time:>20.6f} {gpu_text:>20}")